#  Feature Selection & Model Training

This notebook focuses on selecting the most relevant features and training machine learning models for prediction tasks.

###  Objectives

Build models to predict:

- **Best Delivery App** (Classification)
- **Net Profit** (Regression)

### Workflow Covered

- Importing required libraries  
- Loading processed dataset  
- Categorical encoding  
- Feature selection  
- Train-test split  
- Feature scaling  
- Classification model training  
- Regression model training  
- Model evaluation using performance metrics


## Importing Required Libraries

In this step, all essential Python libraries are imported for data analysis, preprocessing, model building, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             classification_report,
                             mean_absolute_error,
                             mean_squared_error,
                             r2_score)


print("All libraries imported successfully")

All libraries imported successfully


## Loading Processed Dataset

In this step, the cleaned and processed dataset is loaded from the project directory for further analysis and model training.

In [2]:
df = pd.read_csv('..\data\processed\gigsmart_dataset.csv')
print("Dataset loaded")
print("Shape:", df.shape)
df.head(3)

Dataset loaded
Shape: (45593, 16)


,hour,day_of_week,is_weekend,is_festival,weather,zone_type,is_lunch_time,is_dinner_time,distance_km,estimated_time_min,app_name,expected_payout,fuel_cost,time_cost,net_profit,best_app
0,11,5,1,0,clear,normal,0,0,4.2,24,Swiggy,101.55,21.0,36.0,44.55,Swiggy
1,19,4,0,0,rain,low,0,1,14.5,33,Blinkit,220.01,72.5,49.5,98.01,Swiggy
2,8,5,1,0,fog,normal,0,0,6.4,26,Blinkit,104.92,32.0,39.0,33.92,Swiggy


## Categorical Encoding

In this step, categorical columns were converted into numerical format so that machine learning algorithms can process the dataset efficiently.

## One-Hot Encoding Applied On:

- `weather`
- `zone_type`

Separate binary columns were created such as:

- `weather_clear`
- `weather_fog`
- `weather_rain`
- `zone_low`
- `zone_normal`

## Label Encoding Applied On Target Variable:

- `best_app`

Delivery app names were converted into numeric labels:

- Blinkit → 0  
- Swiggy → 1  
- Zomato → 2

## Dataset After Encoding

- Total Rows: **45,593**
- Total Columns: **20**

## Purpose

To transform categorical data into model-ready numerical features for classification and regression tasks.

In [3]:
# One Hot Encoding — weather and zone_type
df_encoded = pd.get_dummies(
    df,
    columns=['weather', 'zone_type'],
    prefix=['weather', 'zone']
)

# Target variable encoded
le_target = LabelEncoder()
df_encoded['best_app'] = le_target.fit_transform(df['best_app'])

print("Target classes:", le_target.classes_)
# Blinkit=0, Swiggy=1, Zomato=2

print("\nEncoding done")
print("Shape after encoding:", df_encoded.shape)
print("Columns:", df_encoded.columns.tolist())

Target classes: ['Blinkit' 'Swiggy' 'Zomato']

Encoding done
Shape after encoding: (45593, 20)
Columns: ['hour', 'day_of_week', 'is_weekend', 'is_festival', 'is_lunch_time', 'is_dinner_time', 'distance_km', 'estimated_time_min', 'app_name', 'expected_payout', 'fuel_cost', 'time_cost', 'net_profit', 'best_app', 'weather_clear', 'weather_fog', 'weather_rain', 'zone_busy', 'zone_low', 'zone_normal']


## Final Feature Selection for Classification & Regression

In this step, separate input features were selected for classification and regression tasks while removing columns that may cause data leakage or target dependency.

## Classification Features (`X_class`)

Selected features used to predict **best_app**:

- Time-based features: `hour`, `day_of_week`, `is_weekend`
- Demand features: `is_festival`, `is_lunch_time`, `is_dinner_time`
- Distance features: `distance_km`, `estimated_time_min`
- Weather encoded features
- Zone encoded features

## Removed from Classification

- `app_name` → Direct leakage risk  
- `expected_payout`, `fuel_cost`, `time_cost` → Profit-related leakage  
- `net_profit` → Separate regression target

## Regression Features (`X_reg`)

Selected features used to predict **net_profit**:

- Time-based features  
- Demand features  
- Distance features  
- Weather encoded features  
- Zone encoded features  

## Removed from Regression

- `expected_payout`, `fuel_cost`, `time_cost` → Direct formula dependency  
- `app_name`, `best_app` → Categorical target leakage

## Targets Defined

- `y_class = best_app`
- `y_reg = net_profit`

## Purpose

To create unbiased and optimized feature sets for accurate model training and better generalization.

In [4]:
# CLASSIFICATION FEATURES
# app_name REMOVE — direct leakage
# expected_payout, fuel_cost, time_cost REMOVE
# net_profit REMOVE — dusra target hai

class_features = [
    'hour', 'day_of_week', 'is_weekend',
    'is_festival', 'is_lunch_time', 'is_dinner_time',
    'distance_km', 'estimated_time_min',
    'weather_clear', 'weather_fog', 'weather_rain',
    'zone_busy', 'zone_low', 'zone_normal'
]

# REGRESSION FEATURES
# expected_payout, fuel_cost, time_cost REMOVE
# kyunki net_profit = payout - fuel - time
# → formula memorize ho jaata hai
# app_name, best_app REMOVE

reg_features = [
    'hour', 'day_of_week', 'is_weekend',
    'is_festival', 'is_lunch_time', 'is_dinner_time',
    'distance_km', 'estimated_time_min',
    'weather_clear', 'weather_fog', 'weather_rain',
    'zone_busy', 'zone_low', 'zone_normal'
]

# Features and targets set 
X_class = df_encoded[class_features]
y_class = df_encoded['best_app']

X_reg = df_encoded[reg_features]
y_reg = df_encoded['net_profit']

print("Classification features:", X_class.columns.tolist())
print("\nClassification target distribution:")
print(y_class.value_counts())
print("\nRegression target stats:")
print(y_reg.describe().round(2))

Classification features: ['hour', 'day_of_week', 'is_weekend', 'is_festival', 'is_lunch_time', 'is_dinner_time', 'distance_km', 'estimated_time_min', 'weather_clear', 'weather_fog', 'weather_rain', 'zone_busy', 'zone_low', 'zone_normal']

Classification target distribution:
best_app
0    16007
2    15794
1    13792
Name: count, dtype: int64

Regression target stats:
count    45593.00
mean        43.54
std         26.40
min        -28.90
25%         24.51
50%         41.48
75%         60.99
max        134.97
Name: net_profit, dtype: float64


## Output Insights

## Classification Target Distribution (`best_app`)

The target classes are fairly balanced:

- Blinkit (0): **16,007**
- Zomato (2): **15,794**
- Swiggy (1): **13,792**

This balanced distribution helps reduce model bias during classification training.

## Regression Target Statistics (`net_profit`)

- Total Records: **45,593**
- Average Profit: **₹43.54**
- Median Profit: **₹41.48**
- Standard Deviation: **₹26.40**

## Profit Range

- Minimum Profit: **₹-28.90** (loss cases)
- Maximum Profit: **₹134.97**

## Quartile Analysis

- 25% orders profit below **₹24.51**
- 50% orders profit below **₹41.48**
- 75% orders profit below **₹60.99**

## Observation

The dataset has a balanced classification target and a realistic spread of profit values, making it suitable for both classification and regression modeling.

## Train-Test Split

The dataset was divided into:

- **80% Training Data** → used to train the models  
- **20% Testing Data** → used to evaluate performance  

`stratify=y_class` was used in classification to maintain balanced class distribution.

`random_state=42` ensures consistent results.

In [5]:
# Classification split — 80/20
X_train_class, X_test_class, \
y_train_class, y_test_class = train_test_split(
    X_class, y_class,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)

# Regression split — 80/20
X_train_reg, X_test_reg, \
y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg,
    test_size=0.20,
    random_state=42
)

print("Classification Split:")
print(f"Train: {X_train_class.shape}")
print(f"Test : {X_test_class.shape}")
print("\nRegression Split:")
print(f"Train: {X_train_reg.shape}")
print(f"Test : {X_test_reg.shape}")

Classification Split:
Train: (36474, 14)
Test : (9119, 14)

Regression Split:
Train: (36474, 14)
Test : (9119, 14)


##  Feature Scaling

`StandardScaler` was applied to standardize feature values for both classification and regression datasets.

- Mean centered to **0**
- Standard deviation scaled to **1**

Training data was fitted first, then the same transformation was applied to test data to avoid data leakage.

In [6]:
scaler = StandardScaler()

# Classification scaling
X_train_class_scaled = scaler.fit_transform(X_train_class)
X_test_class_scaled  = scaler.transform(X_test_class)

# Regression scaling
X_train_reg_scaled = scaler.fit_transform(X_train_reg)
X_test_reg_scaled  = scaler.transform(X_test_reg)

print("Scaling done")

Scaling done


## Classification Model Training

Three classification models were trained to predict the **best delivery app**:

- Logistic Regression  
- Decision Tree Classifier  
- Random Forest Classifier  

## Evaluation Metrics

Models were evaluated using:

- Accuracy  
- Precision  
- Recall  
- F1 Score  

In [7]:


clf_results = {}

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_class_scaled, y_train_class)
y_pred_lr = lr.predict(X_test_class_scaled)

clf_results['Logistic Regression'] = {
    'Accuracy' : accuracy_score(y_test_class, y_pred_lr),
    'Precision': precision_score(y_test_class, y_pred_lr,
                                  average='weighted',
                                  zero_division=0),
    'Recall'   : recall_score(y_test_class, y_pred_lr,
                               average='weighted'),
    'F1 Score' : f1_score(y_test_class, y_pred_lr,
                           average='weighted')
}

# 2. Decision Tree
dt_clf = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
dt_clf.fit(X_train_class, y_train_class)
y_pred_dt = dt_clf.predict(X_test_class)

clf_results['Decision Tree'] = {
    'Accuracy' : accuracy_score(y_test_class, y_pred_dt),
    'Precision': precision_score(y_test_class, y_pred_dt,
                                  average='weighted',
                                  zero_division=0),
    'Recall'   : recall_score(y_test_class, y_pred_dt,
                               average='weighted'),
    'F1 Score' : f1_score(y_test_class, y_pred_dt,
                           average='weighted')
}

# 3. Random Forest
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
rf_clf.fit(X_train_class, y_train_class)
y_pred_rf = rf_clf.predict(X_test_class)

clf_results['Random Forest'] = {
    'Accuracy' : accuracy_score(y_test_class, y_pred_rf),
    'Precision': precision_score(y_test_class, y_pred_rf,
                                  average='weighted',
                                  zero_division=0),
    'Recall'   : recall_score(y_test_class, y_pred_rf,
                               average='weighted'),
    'F1 Score' : f1_score(y_test_class, y_pred_rf,
                           average='weighted')
}

clf_df = pd.DataFrame(clf_results).T.round(4)
print("=" * 55)
print("CLASSIFICATION RESULTS")
print("=" * 55)
print(clf_df)

CLASSIFICATION RESULTS
                     Accuracy  Precision  Recall  F1 Score
Logistic Regression    0.9999     0.9999  0.9999    0.9999
Decision Tree          0.9999     0.9999  0.9999    0.9999
Random Forest          0.9999     0.9999  0.9999    0.9999


## Regression Model Training

Three regression models were trained to predict **net_profit**:

- Linear Regression  
- Decision Tree Regressor  
- Random Forest Regressor  

## Evaluation Metrics

Models were evaluated using:

- MAE (Mean Absolute Error)  
- RMSE (Root Mean Squared Error)  
- R² Score  



In [8]:

reg_results = {}

# 1. Linear Regression 
lin_reg = LinearRegression()
lin_reg.fit(X_train_reg_scaled, y_train_reg)
y_pred_lin = lin_reg.predict(X_test_reg_scaled)

reg_results['Linear Regression'] = {
    'MAE' : mean_absolute_error(y_test_reg, y_pred_lin),
    'RMSE': np.sqrt(mean_squared_error(y_test_reg, y_pred_lin)),
    'R2'  : r2_score(y_test_reg, y_pred_lin)
}

# 2. Decision Tree Regressor
dt_reg = DecisionTreeRegressor(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
dt_reg.fit(X_train_reg, y_train_reg)
y_pred_dt_reg = dt_reg.predict(X_test_reg)

reg_results['Decision Tree'] = {
    'MAE' : mean_absolute_error(y_test_reg, y_pred_dt_reg),
    'RMSE': np.sqrt(mean_squared_error(y_test_reg,
                                        y_pred_dt_reg)),
    'R2'  : r2_score(y_test_reg, y_pred_dt_reg)
}

# 3. Random Forest Regressor
rf_reg = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
rf_reg.fit(X_train_reg, y_train_reg)
y_pred_rf_reg = rf_reg.predict(X_test_reg)

reg_results['Random Forest'] = {
    'MAE' : mean_absolute_error(y_test_reg, y_pred_rf_reg),
    'RMSE': np.sqrt(mean_squared_error(y_test_reg,
                                        y_pred_rf_reg)),
    'R2'  : r2_score(y_test_reg, y_pred_rf_reg)
}

# Results print karo
reg_df = pd.DataFrame(reg_results).T.round(4)
print("=" * 45)
print("REGRESSION RESULTS")
print("=" * 45)
print(reg_df)

REGRESSION RESULTS
                      MAE    RMSE      R2
Linear Regression  4.4878  5.3192  0.9594
Decision Tree      2.4645  3.6316  0.9811
Random Forest      1.8409  2.9578  0.9874
